# Lesson 4.3 — A Goal Pose's Journey to Joint Motion

Trace one pose from decision to movement: pose → configuration → reference → executed motion → arrival, with a check at each stamp.

In [1]:
import numpy as np
from modules.module09.integration import (GreenhouseWorld, model_perception,
    understand, to_configuration, plan_reference, execute_reference,
    velocity_layer, geometric_jacobian, fk_xy, P2, T2)
checks = []
def journey(world, q_start, rng_seed=7, disturbance=None):
    trace = []
    dets = model_perception(world, rng=np.random.default_rng(rng_seed))
    _, tgt = understand(dets, world)
    trace.append(('target pose', '%s %s' % (tgt['id'], np.round(tgt['xy'],3))))
    q_goal = to_configuration(tgt)
    fk_err = float(np.linalg.norm(fk_xy(P2,T2,q_goal) - tgt['xy']))
    trace.append(('goal config', 'IK ok, FK err=%.1e' % fk_err))
    layer = plan_reference(q_start, q_goal, rng=np.random.default_rng(0))
    trace.append(('reference', 'validated=%s, dur=%.2fs' % (layer['validated'], layer['duration'])))
    rec = execute_reference(layer, disturbance=disturbance)
    trace.append(('executed', 'rms=%.4f reached=%s' % (rec['rms'], rec['reached'])))
    arrived = fk_xy(P2, T2, rec['q'][-1])
    trace.append(('arrived', 'FK=%s (target %s)' % (np.round(arrived,3), np.round(tgt['xy'],3))))
    return trace, dict(tgt=tgt, fk_err=fk_err, layer=layer, rec=rec, arrived=arrived)


In [2]:
world = GreenhouseWorld.demo_row(n=6, seed=1)
trace, r = journey(world, world.q.copy())
for s, msg in trace: print(f'{s:13s} | {msg}')
checks.append(r['fk_err'] < 1e-9)                                  # config reaches target
checks.append(r['layer']['validated'])                            # plan validated
checks.append(r['rec']['reached'])                                # execution reached
checks.append(np.linalg.norm(r['arrived'] - r['tgt']['xy']) < 1e-2)  # FK arrives at fruit


target pose   | F3 [0.447 0.152]
goal config   | IK ok, FK err=6.2e-17
reference     | validated=True, dur=1.13s
executed      | rms=0.0001 reached=True
arrived       | FK=[0.447 0.152] (target [0.447 0.152])


### A modest disturbance: the journey still arrives (feedback recovers)

In [3]:
def kick(t, j): return 25.0 if (j == 0 and 0.4 < t < 0.5) else 0.0
trace2, r2 = journey(world, world.q.copy(), disturbance=kick)
print('with disturbance -> reached:', r2['rec']['reached'])
checks.append(r2['rec']['reached'])
assert all(checks), f'FAILED: {checks}'
print(f'{sum(checks)}/{len(checks)} checks passed.')
print('All checks passed.')


with disturbance -> reached: True
5/5 checks passed.
All checks passed.
